# MONARCH · CHAOS — GPU generation notebook (free Colab / Kaggle)

**Yeh notebook aap ke Google account ke GPU par chalta hai.** Main (agent) sirf code aur assets deta hoon; GPU aap ke paas rehta hai, koi credential share nahi hota.

## Kaise chalayein (Colab)
1. Drive → is file par right click → **Open with → Google Colaboratory**
2. Menu: **Runtime → Change runtime type → GPU** (T4 free / A100 with Pro)
3. `Runtime → Run all` — bas.

## Kaise chalayein (Kaggle)
Notebook ko Kaggle par import karein → Settings → Accelerator = **GPU T4 x2** ya **P100** → Run all.
(Kaggle: 30 GPU hours/week free, T4x2 = 2×16 GB)

## Yeh kya banata hai
- `CHAOS_JOBS.json` (Drive folder mein) padhta hai — har job = ek shot.
- Do modes: **`t2v`** (text → cinematic clip) aur **`animate`** (hamara character image + driving video → animated performance, expression replication).
- Har clip `OUT_DIR` mein save + sha256 manifest; already bani clip **skip** ho jaati hai (resume).
- Aakhir mein sab Drive folder mein copy.

## Model (VRAM ke hisaab se automatic)
| VRAM | Model | Note |
|---|---|---|
| ≥ 40 GB (A100/L4x) | `Wan-AI/Wan2.2-Animate-14B-Diffusers` + `Wan2.2-TI2V-5B-Diffusers` | best |
| 22–24 GB (4090/L4) | `Wan-AI/Wan2.2-TI2V-5B-Diffusers` (offload on) | recommended |
| 14–16 GB (T4 free) | `Lightricks/LTX-Video` (2B distilled, low-res) | smoke test / preview |

> **Sach:** free T4 par photoreal 14B models nahi chalenge. T4 = preview + LTX. Final quality ke liye 24 GB+ ya Colab Pro A100.


## 0 · Machine check

In [ ]:
import os, sys, subprocess, json, time, hashlib, pathlib
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")
print("colab:", IN_COLAB, "| kaggle:", IN_KAGGLE)
print(subprocess.run(["python3","-V"],capture_output=True,text=True).stdout.strip())

def gpu_info():
    try:
        out = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version",
                              "--format=csv,noheader"],capture_output=True,text=True).stdout.strip()
        return out or "no nvidia-smi output"
    except FileNotFoundError:
        return "NO GPU DETECTED"

print("GPU:", gpu_info())
import torch
print("torch:", torch.__version__, "| cuda build:", torch.version.cuda, "| available:", torch.cuda.is_available())
VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory/1024**3,1) if torch.cuda.is_available() else 0
print("usable VRAM (GB):", VRAM_GB)
if VRAM_GB == 0:
    raise SystemExit("GPU nahi mila — Runtime → Change runtime type → GPU select karein, phir dobara Run all.")


## 1 · Install (Colab ka torch chhedte nahi)

In [ ]:
%%capture
!pip install -q "diffusers>=0.42.0" "transformers>=4.57" "accelerate>=1.10" safetensors sentencepiece protobuf "imageio-ffmpeg>=0.6" av
!pip install -q hf_transfer || true
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"


In [ ]:
import diffusers, transformers, torch
print("diffusers", diffusers.__version__, "| transformers", transformers.__version__)
names = [n for n in dir(diffusers) if n.endswith("Pipeline")]
for key in ("Wan","LTX","Animate","CogVideo","Hunyuan"):
    print(f"{key}:", sorted([n for n in names if key.lower() in n.lower()])[:8])


## 2 · Drive + folders

`OUT_DIR` = jahan clips banenge. Apna Drive folder ID neeche daal dein (default = `Chaos video 2`).

In [ ]:
DRIVE_FOLDER_ID = "1uGPfl4pwPoL-D0VIag3hIejmbR__ochC"   # Chaos video 2
WORK = pathlib.Path("/content/chaos_gpu")
OUT_DIR = WORK/"clips"; OUT_DIR.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive")
    # try to locate the folder by id using pydrive-free approach: search by name fallback
    print("Drive mounted. Files:", [p.name for p in list(DRIVE_ROOT.iterdir())[:8]])
else:
    DRIVE_ROOT = pathlib.Path("/kaggle/working")

MANIFEST = OUT_DIR/"MANIFEST.json"
manifest = json.loads(MANIFEST.read_text()) if MANIFEST.exists() else {}
print("out:", OUT_DIR, "| existing clips:", len(list(OUT_DIR.glob('*.mp4'))))


## 3 · Jobs file

Agar Drive par `CHAOS_JOBS.json` hai to woh use hoga. Warna ek **smoke-test job** (2 s) chalta hai — sirf yeh check karne ke liye ke GPU path kaam karta hai.

Job schema:
```json
{"jobs":[
  {"id":"A01_wide","mode":"t2v","prompt":"...","seconds":4,"width":720,"height":1280,"seed":1},
  {"id":"C03_face","mode":"animate","image":"images/hero_front.png","driving":"driving/act_C.mp4","seconds":6,"seed":7}
]}
```

In [ ]:
def find_jobs_file():
    cands = [pathlib.Path("/content/drive/MyDrive/CHAOS_JOBS.json"),
             pathlib.Path("/content/drive/MyDrive/Chaos video 2/CHAOS_JOBS.json"),
             pathlib.Path("/kaggle/input/chaos-jobs/CHAOS_JOBS.json"),
             WORK/"CHAOS_JOBS.json"]
    for c in cands:
        if c.exists():
            return c
    return None

jobs_path = find_jobs_file()
if jobs_path:
    JOBS = json.loads(jobs_path.read_text())["jobs"]
    print("jobs from", jobs_path, "->", len(JOBS))
else:
    JOBS = [{"id":"smoke_test","mode":"t2v","seed":1,"seconds":2,"width":480,"height":854,
             "prompt":"dark industrial hall, red portal light, heavy haze, cinematic vertical, no text, no watermark"}]
    print("no CHAOS_JOBS.json found -> smoke test only")
print(json.dumps(JOBS[:2], indent=1)[:800])


## 4 · Generators (VRAM ke hisaab se auto-select)

In [ ]:
import gc, torch
from diffusers.utils import export_to_video

def vram():
    return torch.cuda.get_device_properties(0).total_memory/1024**3 if torch.cuda.is_available() else 0

_pipe = {}

def get_ltx():
    if "ltx" not in _pipe:
        from diffusers import LTXPipeline
        # 2B distilled checkpoint = lowest VRAM path (works on T4 with offload)
        try:
            _pipe["ltx"] = LTXPipeline.from_pretrained("Lightricks/LTX-Video", torch_dtype=torch.bfloat16)
        except Exception as e:
            print("LTX load fallback (fp16):", e)
            _pipe["ltx"] = LTXPipeline.from_pretrained("Lightricks/LTX-Video", torch_dtype=torch.float16)
        _pipe["ltx"].enable_model_cpu_offload()
    return _pipe["ltx"]

def get_wan5b():
    if "wan" not in _pipe:
        from diffusers import WanPipeline
        _pipe["wan"] = WanPipeline.from_pretrained("Wan-AI/Wan2.2-TI2V-5B-Diffusers",
                                                   torch_dtype=torch.bfloat16)
        _pipe["wan"].enable_model_cpu_offload()
    return _pipe["wan"]

def get_animate():
    if "animate" not in _pipe:
        import diffusers
        cls = None
        for n in dir(diffusers):
            if "animate" in n.lower() and n.endswith("Pipeline"):
                cls = getattr(diffusers, n); break
        if cls is None:
            raise RuntimeError("Is diffusers build mein Wan-Animate pipeline nahi hai — `pip install -U diffusers` karein.")
        _pipe["animate"] = cls.from_pretrained("Wan-AI/Wan2.2-Animate-14B-Diffusers",
                                               torch_dtype=torch.bfloat16)
        _pipe["animate"].enable_model_cpu_offload()
    return _pipe["animate"]

def pick_engine(job):
    if vram() >= 38:
        return "animate" if job["mode"] == "animate" else "wan5b"
    if vram() >= 20:
        return "wan5b"
    return "ltx"

def gen(job, out_path):
    eng = pick_engine(job)
    g = torch.Generator(device="cpu").manual_seed(int(job.get("seed", 0)))
    fps = 24
    n_frames = max(25, int(round(job.get("seconds", 4) * fps)) // 4 * 4 + 1)
    print(f"  engine={eng} frames={n_frames} size={job.get('width',704)}x{job.get('height',1280)}")
    if eng == "ltx":
        pipe = get_ltx()
        frames = pipe(prompt=job["prompt"], width=min(job.get("width",704), 768),
                      height=min(job.get("height",1280), 1280), num_frames=min(n_frames, 121),
                      num_inference_steps=int(job.get("steps", 30)),
                      guidance_scale=float(job.get("cfg", 3.0)), generator=g).frames[0]
    elif eng == "wan5b":
        pipe = get_wan5b()
        frames = pipe(prompt=job["prompt"], height=job.get("height",1280), width=job.get("width",704),
                      num_frames=n_frames, num_inference_steps=int(job.get("steps", 30)),
                      guidance_scale=float(job.get("cfg", 5.0)), generator=g).frames[0]
    else:
        from PIL import Image
        pipe = get_animate()
        img = Image.open(job["image"]).convert("RGB")
        frames = pipe(image=img, video=job["driving"], prompt=job.get("prompt",""),
                      num_inference_steps=int(job.get("steps", 20)),
                      generator=g).frames[0]
    export_to_video(frames, str(out_path), fps=fps)
    return eng


## 5 · Run (resume + manifest)

In [ ]:
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

results = []
for job in JOBS:
    jid = job["id"]; out = OUT_DIR/f"{jid}.mp4"
    if out.exists() and out.stat().st_size > 20000:
        print(f"[skip] {jid} already exists"); 
        results.append({"id":jid,"file":out.name,"status":"existing","sha256":sha256(out)}); continue
    t0 = time.time()
    try:
        eng = gen(job, out)
        entry = {"id":jid,"file":out.name,"status":"ok","engine":eng,
                 "seconds":round(time.time()-t0,1),"bytes":out.stat().st_size,"sha256":sha256(out)}
        print(f"[ok] {jid} {entry['seconds']}s {entry['bytes']:,}B")
    except Exception as e:
        entry = {"id":jid,"status":"error","error":f"{type(e).__name__}: {e}"}
        print(f"[ERR] {jid} -> {entry['error'][:300]}")
        if "out of memory" in entry["error"].lower():
            print("  hint: width/height/frames kam karein ya 24GB+ GPU (Colab Pro A100) use karein")
            torch.cuda.empty_cache(); gc.collect()
    results.append(entry)
    manifest[jid] = entry
    MANIFEST.write_text(json.dumps(manifest, indent=2))
print("\nDONE:", sum(1 for r in results if r.get("status") in ("ok","existing")), "/", len(results))


## 6 · Drive par wapas bhejo

In [ ]:
if IN_COLAB:
    from google.colab import files
    target_name = "chaos_gpu_clips"
    tgt = DRIVE_ROOT/target_name
    tgt.mkdir(parents=True, exist_ok=True)
    import shutil
    n = 0
    for f in OUT_DIR.glob("*.mp4"):
        shutil.copy2(f, tgt/f.name); n += 1
    (tgt/"MANIFEST.json").write_text(json.dumps(manifest, indent=2))
    print(f"{n} clips + MANIFEST.json -> {tgt}")
    print("Ab agent ko bolein: clips Drive ke is folder mein hain — woh edit/grade/QC/delivery karega.")
else:
    print("Kaggle: /kaggle/working se Output tab mein download karein.")
